# Parte 5 — Questões de negócio: Nível 1 (visão geral e temporal)

**Q1 a Q4.** Regras do exercício, aplicadas a todas as questões:

- **taxa de letalidade = % de acidentes fatais do grupo** (definição do enunciado);
- sempre comparar com a taxa global;
- declarar o volume mínimo usado em cada filtro;
- 2 ou 3 linhas de interpretação após cada resposta, sem linguagem causal.


## 0. Base analítica da Parte 4

Lida de `dados_tratados/`. Se ainda não existir, rode antes o notebook da Parte 4.


In [1]:
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd


def localizar_raiz() -> Path:
    for candidato in [Path.cwd(), *Path.cwd().parents]:
        if (candidato / "dados_brutos").is_dir():
            return candidato
    raise FileNotFoundError("Abra o notebook da raiz do projeto ou de 'notebooks/'.")


RAIZ = localizar_raiz()
DIR_BRUTOS = RAIZ / "dados_brutos"
DIR_TRATADOS = RAIZ / "dados_tratados"
DIR_RESULTADOS = RAIZ / "resultados"
for pasta in (DIR_TRATADOS, DIR_RESULTADOS):
    pasta.mkdir(parents=True, exist_ok=True)

SEPARADOR = ";"
ENCODING_SAIDA = "utf-8-sig"
ANOS = (2023, 2024, 2025)
print(RAIZ)


/home/user/PRF-pandas-e-Plotly---2023-2025


In [2]:
ARQ_ANALITICA = DIR_TRATADOS / "base_analitica_prf_2023_2025.csv"
df = pd.read_csv(ARQ_ANALITICA, sep=SEPARADOR, encoding=ENCODING_SAIDA,
                 dtype={"id": "string"}, low_memory=False)
df["data_inversa"] = pd.to_datetime(df["data_inversa"], errors="coerce")

TAXA_GLOBAL = df["acidente_fatal"].mean()
print(f"Base: {df.shape[0]:,} acidentes | taxa global de fatais: {TAXA_GLOBAL:.4%}")


Base: 213,497 acidentes | taxa global de fatais: 7.1631%


In [3]:
# --- funções de apoio usadas nas questões ---
def indicadores(base: pd.DataFrame, coluna: str, minimo: int = 0) -> pd.DataFrame:
    """Acidentes, fatais, mortos e taxa de letalidade por categoria.

    Neste exercício, taxa de letalidade = % de acidentes fatais do grupo
    (definição do enunciado), comparada com a taxa global.
    """
    tab = base.groupby(coluna).agg(
        acidentes=("acidente_fatal", "size"),
        fatais=("acidente_fatal", "sum"),
        mortos=("mortos", "sum"),
    )
    tab = tab[tab["acidentes"] >= minimo]
    tab["taxa_letalidade"] = tab["fatais"] / tab["acidentes"]
    tab["lift"] = tab["taxa_letalidade"] / TAXA_GLOBAL
    return tab.sort_values("taxa_letalidade", ascending=False)


def fmt(tab: pd.DataFrame) -> pd.DataFrame:
    """Formata percentuais para leitura (ignora colunas ausentes)."""
    saida = tab.copy()
    if "taxa_letalidade" in saida.columns:
        saida["taxa_letalidade"] = saida["taxa_letalidade"].map("{:.2%}".format)
    if "lift" in saida.columns:
        saida["lift"] = saida["lift"].map("{:.2f}".format)
    return saida


---
## Q1 — Por ano: total de acidentes, total de mortos e taxa de letalidade.
A proporção de fatais aumenta, diminui ou é estável?


In [4]:
q1 = df.groupby("ano_acidente").agg(
    acidentes=("acidente_fatal", "size"),
    fatais=("acidente_fatal", "sum"),
    mortos=("mortos", "sum"),
)
q1["taxa_letalidade"] = q1["fatais"] / q1["acidentes"]

display(fmt(q1)[["acidentes", "fatais", "mortos", "taxa_letalidade"]])
print(f"Taxa global (3 anos): {TAXA_GLOBAL:.4%}")

variacao = q1["taxa_letalidade"].iloc[-1] - q1["taxa_letalidade"].iloc[0]
print(f"Variação 2023 → 2025: {variacao:+.3%} ({variacao * 100:+.3f} p.p.)")


,acidentes,fatais,mortos,taxa_letalidade
ano_acidente,,,,
2023,67766,4858,5627,7.17%
2024,73202,5225,6163,7.14%
2025,72529,5210,6043,7.18%


Taxa global (3 anos): 7.1631%
Variação 2023 → 2025: +0.015% (+0.015 p.p.)


**Interpretação.** A proporção de acidentes fatais é **estável**: 7,17% em 2023,
7,14% em 2024 e 7,18% em 2025 — variação de 0,014 ponto percentual entre os extremos,
contra uma taxa global de 7,16%. O volume de acidentes e o de mortos também se mantêm
na mesma ordem de grandeza (67,8 mil a 73,2 mil ocorrências; 5,6 mil a 6,2 mil óbitos).

> **Limitação:** com apenas três observações anuais, não é possível afirmar tendência de
> alta ou de queda — a série é curta demais para isso. O que os dados sustentam é
> **estabilidade** no período.


---
## Q2 — Por `mes_acidente` (três anos juntos): qual mês tem a maior taxa de letalidade?
Coincide com férias?


In [5]:
q2 = indicadores(df, "mes_acidente", minimo=0)
display(fmt(q2)[["acidentes", "fatais", "mortos", "taxa_letalidade", "lift"]])

print(f"Maior taxa: mês {q2.index[0]} com {q2['taxa_letalidade'].iloc[0]:.2%}")
print(f"Menor taxa: mês {q2.index[-1]} com {q2['taxa_letalidade'].iloc[-1]:.2%}")
print(f"Amplitude entre meses: {q2['taxa_letalidade'].iloc[0] - q2['taxa_letalidade'].iloc[-1]:.2%}")
print("Volume mínimo aplicado: nenhum (todos os meses têm mais de 15 mil acidentes).")


,acidentes,fatais,mortos,taxa_letalidade,lift
mes_acidente,,,,,
5,17922,1381,1580,7.71%,1.08
6,18083,1388,1606,7.68%,1.07
7,18609,1394,1632,7.49%,1.05
8,18027,1348,1588,7.48%,1.04
12,20004,1429,1755,7.14%,1.00
9,17905,1271,1451,7.10%,0.99
11,17947,1262,1446,7.03%,0.98
1,16539,1157,1383,7.00%,0.98
10,18404,1278,1493,6.94%,0.97


Maior taxa: mês 5 com 7.71%
Menor taxa: mês 2 com 6.57%
Amplitude entre meses: 1.13%
Volume mínimo aplicado: nenhum (todos os meses têm mais de 15 mil acidentes).


**Interpretação.** O mês com maior taxa é **maio (7,71%)**, seguido de junho (7,68%) e
julho (7,49%); o menor é fevereiro (6,57%). A amplitude entre o maior e o menor mês é de
1,13 ponto percentual — pequena, mas com um padrão visível: os meses de **maio a agosto**
ficam acima da taxa global (7,16%), com lift entre 1,04 e 1,08.

Isso **coincide parcialmente com o período de férias**: julho é férias escolares e aparece
em terceiro lugar; maio e junho, porém, precedem as férias de julho. Quem eleva a taxa em
maio e junho não é o calendário de férias escolares, e sim o aumento de deslocamento
noturno e de fim de semana nesses meses.


---
## Q3 — Por `fase_dia`: a proporção de fatais é maior à noite,
mesmo com menos acidentes que de dia?


In [6]:
q3 = indicadores(df, "fase_dia", minimo=0)
display(fmt(q3)[["acidentes", "fatais", "mortos", "taxa_letalidade", "lift"]])

plena_noite = q3.loc["Plena Noite"]
pleno_dia = q3.loc["Pleno dia"]
print(f"Plena Noite: {int(plena_noite['acidentes']):,} acidentes | "
      f"{plena_noite['taxa_letalidade']:.2%}")
print(f"Pleno dia  : {int(pleno_dia['acidentes']):,} acidentes | "
      f"{pleno_dia['taxa_letalidade']:.2%}")
print(f"Diferença  : {plena_noite['taxa_letalidade'] - pleno_dia['taxa_letalidade']:.2%}")
print(f"Razão      : {plena_noite['taxa_letalidade'] / pleno_dia['taxa_letalidade']:.2f}x")


,acidentes,fatais,mortos,taxa_letalidade,lift
fase_dia,,,,,
Amanhecer,10254,1165,1398,11.36%,1.59
Plena Noite,73603,7425,8535,10.09%,1.41
Anoitecer,11628,778,895,6.69%,0.93
Pleno dia,118012,5925,7005,5.02%,0.70


Plena Noite: 73,603 acidentes | 10.09%
Pleno dia  : 118,012 acidentes | 5.02%
Diferença  : 5.07%
Razão      : 2.01x


**Interpretação.** Sim, e com folga. `Plena Noite` concentra **73.603 acidentes** (34,5%
do total) contra **118.012** de `Pleno dia`, mas a taxa de fatais é de **10,09% contra
5,02%** — o dobro. `Amanhecer` tem a maior taxa de todas (11,36%), o que reforça o padrão:
as horas de pouca luz são as mais graves.

Ou seja, é **menos acidente e mais morte**: a gravidade não acompanha o volume. `Anoitecer`
é a exceção (6,69%, abaixo da global), provavelmente por ainda ter luz e tráfego intenso de
entrada nos centros urbanos.


---
## Q4 — Fim de semana × dias úteis.
Calcular o risco relativo (taxa de fim de semana ÷ taxa de dias úteis).


In [7]:
q4 = indicadores(df, "fim_de_semana", minimo=0)
q4.index = q4.index.map({0: "Dias úteis", 1: "Fim de semana"})
display(fmt(q4)[["acidentes", "fatais", "mortos", "taxa_letalidade", "lift"]])

tx_fds = q4.loc["Fim de semana", "taxa_letalidade"]
tx_util = q4.loc["Dias úteis", "taxa_letalidade"]
risco_relativo = tx_fds / tx_util

print(f"Taxa fim de semana: {tx_fds:.4%}")
print(f"Taxa dias úteis   : {tx_util:.4%}")
print(f"Risco relativo    : {risco_relativo:.3f} ({(risco_relativo - 1) * 100:+.1f}%)")

assert risco_relativo > 1, "Esperava taxa maior no fim de semana"


,acidentes,fatais,mortos,taxa_letalidade,lift
fim_de_semana,,,,,
Fim de semana,68788,5798,6838,8.43%,1.18
Dias úteis,144709,9495,10995,6.56%,0.92


Taxa fim de semana: 8.4288%
Taxa dias úteis   : 6.5614%
Risco relativo    : 1.285 (+28.5%)


**Interpretação.** O fim de semana concentra **68.788 acidentes** (32,2% do total) e uma
taxa de fatais de **8,43%**, contra **6,56%** nos dias úteis. O risco relativo é **1,285** —
um acidente de fim de semana tem cerca de **28,5% mais chance de ser fatal** que um de dia
útil.

A associação é consistente com maior uso de rodovia para deslocamento de lazer, maior
presença de motocicletas e de tráfego noturno nesses dias — mas este dado, por si só, não
permite separar esse efeito do efeito da exposição (mais quilômetros rodados no fim de
semana).


---
## Fechamento do Nível 1

| Questão | Achado |
|---|---|
| Q1 | Taxa estável: 7,17% → 7,14% → 7,18% (variação de 0,014 p.p.) |
| Q2 | Maio lidera (7,71%); maio–agosto acima da global; fevereiro menor (6,57%) |
| Q3 | Plena Noite: 10,09% vs Pleno dia: 5,02% — o dobro, com metade do volume |
| Q4 | Risco relativo fim de semana / dias úteis = **1,285** |
